In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")

In [ ]:
df_ground_truth.head()

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,"If I sign up now, is it still possible to get ...",74eb249bbf
2,What do I need to do to be eligible for the ce...,74eb249bbf
3,Is there a deadline for submitting the project...,74eb249bbf
4,Can late joiners still take part in the course...,74eb249bbf


In [ ]:

ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop stream if the Zoom link isn’t shared publicly?',
 'document': '489dd1c9d9'}

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': 'db78580409

In [ ]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
q = ground_truth[0]
q

{'question': 'I just found this course late — can I still join in, or is it too late to start?',
 'document': '74eb249bbf'}

In [ ]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [ ]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](htt

In [ ]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [ ]:
text_search(q["question"])

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](htt

In [ ]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [ ]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course late — can I still join in, or is it too late to start?


[1, 0, 0, 0, 0]

In [ ]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where should I look for the live session video link before an Office Hours starts?


[1, 0, 0, 0, 0]

In [ ]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

How do I keep track of the LLM Zoomcamp syllabus and what’s due each week?


[1, 0, 0, 0, 0]

In [ ]:
[0, 0, 0, 0, 0]

[0, 0, 0, 0, 0]

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
relevance[:15]

[1, 0, 0, 0, 0]

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
sample = relevance_total = compute_relevance_total(ground_truth_sample, text_search)


  0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

In [ ]:

sample = relevance_total[:15]

In [ ]:
sample

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [ ]:
14 / 15

0.9333333333333333

## Search Evaluation Metrics

In [ ]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
hit_rate

<function __main__.hit_rate(relevance)>

In [ ]:
hit_rate(relevance_total)

0.8288135593220339

In [ ]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.8

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [ ]:
mrr(sample)

0.8

In [ ]:
	
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8288135593220339, 'mrr': 0.7144067796610165}

In [ ]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8491525423728814, 'mrr': 0.7298022598870052}

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/590 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8762711864406779, 'mrr': 0.7709039548022594}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8796610169491526, 'mrr': 0.769604519774011}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8288135593220339, 'mrr': 0.7144067796610165}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8, 'mrr': 0.6884180790960448}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7745762711864407, 'mrr': 0.6565819209039544}


In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/590 [00:00<?, ?it/s]

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.969492,0.866582
6,1.0,4.0,0.1,0.969492,0.863559
35,5.0,10.0,0.5,0.967797,0.860311
19,2.0,4.0,0.2,0.967797,0.860311
3,1.0,2.0,0.1,0.967797,0.860311
8,1.0,4.0,0.5,0.961017,0.860198
34,5.0,10.0,0.2,0.967797,0.858898
22,2.0,10.0,0.2,0.967797,0.857994
33,5.0,10.0,0.1,0.967797,0.857910
18,2.0,4.0,0.1,0.967797,0.857853


In [ ]:
document_ids = {
    str(doc["id"]).strip()
    for doc in documents
}

ground_truth_ids = set(df_ground_truth["document"])

missing_ids = ground_truth_ids - document_ids

print("Fehlende IDs:", len(missing_ids))
print("Beispiele:", sorted(missing_ids)[:10])

Fehlende IDs: 0
Beispiele: []
